# EDA ejecutivo y oportunidades de captación

## tl;dr

Con los umbrales bloqueados se identifican **29 segmentos candidatos** en cinco de las
seis ciudades; Londres no conserva candidatos tras las sensibilidades. Son prioridades
provisionales para investigar captación, no estimaciones de demanda, reservas, ocupación
o margen.

## Contexto y métodos

### Pregunta de decisión

¿Qué tipologías conviene investigar primero para captar nuevos anfitriones y en qué
barrios? La unidad es `ciudad + barrio + tipología`. Se comparan actividad histórica,
cuota de oferta, tamaño, precio local y evidencia estadística por separado.

### Supuestos clave

La tasa de reseñas es un proxy histórico. Todos los contrastes y precios se interpretan
dentro de ciudad. Se usan alfa 0,05, IC 95 %, efectos, correcciones Holm/BH y
sensibilidades de casos completos, outliers y concentración por anfitrión.

### 1. Cargar resultados aceptados

In [ ]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import display
from airbnb_supply_analysis.visualization import (
    activity_by_room_type,
    association_effects,
    opportunity_scatter,
)

ROOT = Path("..").resolve()
PROCESSED = Path(os.environ.get("AIRBNB_SUPPLY_PROCESSED_DIR", ROOT / "data/processed"))
listings = pd.read_parquet(PROCESSED / "listings.parquet")
results = pd.read_parquet(PROCESSED / "statistical_results.parquet")
segments = pd.read_parquet(PROCESSED / "opportunity_segments.parquet")
{"anuncios": len(listings), "segmentos": len(segments), "resultados": len(results)}

**Conclusión.** Los tres artefactos comparten build y derivan de 220.031 anuncios. La
cobertura permite comparar patrones internos, pero no garantiza representatividad del
mercado completo.

### 2. Distribución de actividad por tipología

In [ ]:
room_summary = (
    listings.dropna(subset=["room_type", "activity_proxy"])
    .groupby(["city_key", "room_type"], observed=True)
    .agg(
        anuncios=("listing_key", "size"),
        actividad_mediana=("activity_proxy", "median"),
        cuota_positiva=("activity_proxy", lambda values: values.gt(0).mean()),
    )
    .reset_index()
)
display(room_summary)
activity_by_room_type(listings)

**Conclusión.** Las tipologías difieren en escala, mediana y probabilidad de actividad.
Una mediana mayor no basta para recomendar captación: se exige contraste dentro de
ciudad, efecto, precisión, corrección y cuota relativa de oferta.

### 3. Contrastes y asociaciones dentro de ciudad

In [ ]:
room_tests = results.query("method == 'kruskal_wallis'")[
    ["city_key", "sample_size", "estimate", "p_value_adjusted"]
]
associations = results.query("method == 'spearman'")[
    ["city_key", "metric", "estimate", "ci_low", "ci_high", "p_value_adjusted"]
]
display(room_tests)
display(associations)
association_effects(results)

**Conclusión.** La relación entre precio publicado y actividad es débil en todas las
ciudades observadas; cambia de signo en Tokio. Las noches mínimas muestran asociaciones
negativas de magnitud variable. Aunque muchos valores ajustados son pequeños por el gran
tamaño muestral, las correlaciones no implican causalidad ni rentabilidad.

### 4. Matriz de oportunidades y top tres por ciudad

In [ ]:
candidates = segments.query("opportunity_label == 'candidate'").copy()
candidate_counts = candidates.groupby("city_key", observed=True).size()
top_columns = [
    "city_key", "neighborhood", "room_type", "listing_count",
    "activity_median", "probability_superiority", "effect_ci_low",
    "q_value", "neighborhood_room_type_share", "room_type_city_share",
    "candidate_rank",
]
top_candidates = (
    candidates.sort_values(["city_key", "candidate_rank"])[top_columns]
    .groupby("city_key", observed=True)
    .head(3)
)
display(candidate_counts.to_frame("segmentos_candidatos"))
display(top_candidates)

**Conclusión.** El primer foco por escala es Justicia-habitación privada en Madrid,
CENTRALE-alojamiento completo en Milán, Bedford-Stuyvesant-alojamiento completo en Nueva
York, Leichhardt-habitación privada en Sídney y Nakano Ku-habitación privada en Tokio.
Londres queda sin candidato robusto; no se rebajan reglas para forzar un top tres.

### 5. Explorar actividad relativa frente a cuota local

In [ ]:
opportunity_scatter(segments)

**Conclusión.** La oportunidad aparente combina evidencia de actividad superior y cuota
local inferior a la ciudad, manteniendo escala y precisión visibles. El gráfico permite
explorar excepciones, pero la etiqueta procede de reglas versionadas y no de selección
visual.

## Takeaways

Se recomienda investigar primero los candidatos mostrados y validar la oportunidad con
búsquedas, reservas, ocupación, conversión, ingresos y capacidad real de captación. Los
resultados actuales sirven para priorizar investigación comercial; no justifican una
expansión automática ni una promesa de margen.